# Binary & Multi-Class Classification

## Food 101

Importing all required dataset, packages & libraries

In [9]:
import pandas as pd
import numpy as np
import joblib
import os
import random
import shutil
import torch
from collections import defaultdict
from constants import RANDOM_SEED
from utils import set_seed
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from torch.utils.data import Subset
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

In [10]:
# --- Paths ---
FOOD101_ROOT = "D:/CitrusBits/pythonic-rebirth/datasets/food-101/food-101"
IMAGES_DIR = os.path.join(FOOD101_ROOT, "images")
META_DIR = os.path.join(FOOD101_ROOT, "meta")

TRAIN_TXT = os.path.join(META_DIR, "train.txt")
TEST_TXT = os.path.join(META_DIR, "test.txt")

# selecting 15 classes out of 101
SELECTED_CLASSES = [
    'pizza', 'sushi', 'ice_cream', 'hamburger', 'ramen',
    'tacos', 'donuts', 'french_fries', 'chicken_wings', 'steak',
    'apple_pie', 'caesar_salad', 'pancakes', 'spaghetti_bolognese', 'waffles'
]
N_TRAIN_PER_CLASS = 200
N_TEST_PER_CLASS = 60

random.seed(RANDOM_SEED)


# --- Read official train/test split files ---
# Each line in train.txt/test.txt looks like: "pizza/1234567"  (no file extension)
def read_split_file(path):
    class_to_files = defaultdict(list)
    with open(path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            class_name, file_id = line.split('/')
            class_to_files[class_name].append(file_id)
    return class_to_files


train_split = read_split_file(TRAIN_TXT)
test_split = read_split_file(TEST_TXT)

# --- Build a smaller working dataset by copying selected images into a new folder ---
SUBSET_DIR = os.path.join(FOOD101_ROOT, "subset")
os.makedirs(SUBSET_DIR, exist_ok=True)


def build_subset(split_dict, class_list, n_per_class, split_name):
    dest_root = os.path.join(SUBSET_DIR, split_name)
    for class_name in class_list:
        available = split_dict[class_name]
        sampled = random.sample(available, min(n_per_class, len(available)))

        dest_dir = os.path.join(dest_root, class_name)
        os.makedirs(dest_dir, exist_ok=True)

        for file_id in sampled:
            src = os.path.join(IMAGES_DIR, class_name, f"{file_id}.jpg")
            dst = os.path.join(dest_dir, f"{file_id}.jpg")
            if not os.path.exists(dst):  # avoid re-copying if rerun
                shutil.copy2(src, dst)

    print(f"Built '{split_name}' subset: {len(class_list)} classes x up to {n_per_class} images")


build_subset(train_split, SELECTED_CLASSES, N_TRAIN_PER_CLASS, "train")
build_subset(test_split, SELECTED_CLASSES, N_TEST_PER_CLASS, "test")

print(f"Subset created at: {SUBSET_DIR}")

Built 'train' subset: 15 classes x up to 200 images
Built 'test' subset: 15 classes x up to 60 images
Subset created at: D:/CitrusBits/pythonic-rebirth/datasets/food-101/food-101\subset


In [11]:
SUBSET_DIR = "D:/CitrusBits/pythonic-rebirth/datasets/food-101/food-101/subset"
IMG_SIZE = 128

# --- Transforms ---
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),  # mild rotation aug — food photos aren't always perfectly upright
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

eval_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# --- Datasets ---
full_train_dataset = datasets.ImageFolder(os.path.join(SUBSET_DIR, "train"), transform=train_transforms)
test_dataset = datasets.ImageFolder(os.path.join(SUBSET_DIR, "test"), transform=eval_transforms)

print("Classes:", full_train_dataset.classes)
print("Class-to-index mapping:", full_train_dataset.class_to_idx)
print(f"Full train size: {len(full_train_dataset)}")
print(f"Test size: {len(test_dataset)}")

# --- Carve out a small validation split from train (Food-101 only ships train/test) ---
# from constants import RANDOM_SEED
generator = torch.Generator().manual_seed(RANDOM_SEED)

val_fraction = 0.1
val_size = int(len(full_train_dataset) * val_fraction)
train_size = len(full_train_dataset) - val_size

train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size], generator=generator)

print(f"Train size after val split: {train_size}")
print(f"Val size: {val_size}")

# --- DataLoaders ---
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# --- Confirm class balance ---
from collections import Counter

train_labels = [full_train_dataset.targets[i] for i in train_dataset.indices]
print("Train class distribution:", Counter(train_labels))

Classes: ['apple_pie', 'caesar_salad', 'chicken_wings', 'donuts', 'french_fries', 'hamburger', 'ice_cream', 'pancakes', 'pizza', 'ramen', 'spaghetti_bolognese', 'steak', 'sushi', 'tacos', 'waffles']
Class-to-index mapping: {'apple_pie': 0, 'caesar_salad': 1, 'chicken_wings': 2, 'donuts': 3, 'french_fries': 4, 'hamburger': 5, 'ice_cream': 6, 'pancakes': 7, 'pizza': 8, 'ramen': 9, 'spaghetti_bolognese': 10, 'steak': 11, 'sushi': 12, 'tacos': 13, 'waffles': 14}
Full train size: 3000
Test size: 900
Train size after val split: 2700
Val size: 300
Train class distribution: Counter({3: 190, 7: 185, 14: 184, 8: 183, 1: 183, 4: 182, 10: 181, 11: 179, 2: 178, 13: 178, 12: 177, 0: 177, 9: 176, 6: 175, 5: 172})


In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# --- CNN Architecture for 15-class food classification ---
class Food101CNN(nn.Module):
    def __init__(self, num_classes=15):
        super(Food101CNN, self).__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 128 -> 64

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 64 -> 32

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 32 -> 16

            # Block 4
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 16 -> 8
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(256, 256),  # wider than X-Ray's 128, since 15-way needs more representational capacity
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = Food101CNN(num_classes=15).to(device)

# --- No class weighting needed — dataset is naturally balanced ---
criterion = nn.CrossEntropyLoss()

# --- Lower LR from the start, given what we learned from X-Ray ---
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# --- Save/load setup ---
MODEL_DIR = "D:/CitrusBits/pythonic-rebirth/models"
os.makedirs(MODEL_DIR, exist_ok=True)
CNN_PATH = os.path.join(MODEL_DIR, "food101_cnn_scratch.pth")

EPOCHS = 20

if os.path.exists(CNN_PATH):
    print(f"Found existing model at {CNN_PATH}, loading instead of retraining...")
    model.load_state_dict(torch.load(CNN_PATH, map_location=device))
else:
    print("No saved model found, training CNN from scratch...")
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        correct = 0
        total_samples = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

        epoch_loss = running_loss / total_samples
        epoch_acc = correct / total_samples

        # Validation phase — every epoch
        model.eval()
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs, 1)
                val_correct += (predicted == labels).sum().item()
                val_total += labels.size(0)
        val_acc = val_correct / val_total

        print(
            f"Epoch {epoch + 1}/{EPOCHS} — Train Loss: {epoch_loss:.4f}, Train Acc: {epoch_acc:.4f}, Val Acc: {val_acc:.4f}")

    torch.save(model.state_dict(), CNN_PATH)
    print(f"Model saved to {CNN_PATH}")

Using device: cpu
No saved model found, training CNN from scratch...
Epoch 1/20 — Train Loss: 2.6642, Train Acc: 0.1070, Val Acc: 0.1667
Epoch 2/20 — Train Loss: 2.5800, Train Acc: 0.1667, Val Acc: 0.1967
Epoch 3/20 — Train Loss: 2.5032, Train Acc: 0.1878, Val Acc: 0.1900
Epoch 4/20 — Train Loss: 2.4608, Train Acc: 0.2033, Val Acc: 0.2200
Epoch 5/20 — Train Loss: 2.4362, Train Acc: 0.2063, Val Acc: 0.2467
Epoch 6/20 — Train Loss: 2.3922, Train Acc: 0.2330, Val Acc: 0.2667
Epoch 7/20 — Train Loss: 2.3562, Train Acc: 0.2474, Val Acc: 0.2767
Epoch 8/20 — Train Loss: 2.3371, Train Acc: 0.2459, Val Acc: 0.2500
Epoch 9/20 — Train Loss: 2.3059, Train Acc: 0.2611, Val Acc: 0.2700
Epoch 10/20 — Train Loss: 2.2612, Train Acc: 0.2737, Val Acc: 0.2500
Epoch 11/20 — Train Loss: 2.2376, Train Acc: 0.2678, Val Acc: 0.2533
Epoch 12/20 — Train Loss: 2.2078, Train Acc: 0.2878, Val Acc: 0.2467
Epoch 13/20 — Train Loss: 2.1933, Train Acc: 0.2893, Val Acc: 0.3133
Epoch 14/20 — Train Loss: 2.1690, Train Acc

In [13]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

model.eval()
correct = 0
total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_acc = correct / total
print(f"Test Accuracy: {test_acc:.4f}")

class_names = full_train_dataset.classes  # the 15 food class names, in correct index order

print(classification_report(all_labels, all_preds, target_names=class_names))
print(confusion_matrix(all_labels, all_preds))

Test Accuracy: 0.3689
                     precision    recall  f1-score   support

          apple_pie       0.29      0.20      0.24        60
       caesar_salad       0.42      0.82      0.56        60
      chicken_wings       0.35      0.63      0.45        60
             donuts       0.36      0.22      0.27        60
       french_fries       0.47      0.62      0.54        60
          hamburger       0.14      0.02      0.03        60
          ice_cream       0.28      0.53      0.36        60
           pancakes       0.50      0.20      0.29        60
              pizza       0.66      0.35      0.46        60
              ramen       0.33      0.25      0.29        60
spaghetti_bolognese       0.60      0.48      0.54        60
              steak       0.31      0.67      0.43        60
              sushi       0.26      0.30      0.28        60
              tacos       0.30      0.13      0.18        60
            waffles       0.29      0.12      0.17        60



In [14]:
# now lets use a pretrained model
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader, random_split
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- New transforms at 224x224 for the pretrained model ---
IMG_SIZE_PRETRAINED = 224

train_transforms_pt = transforms.Compose([
    transforms.Resize((IMG_SIZE_PRETRAINED, IMG_SIZE_PRETRAINED)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

eval_transforms_pt = transforms.Compose([
    transforms.Resize((IMG_SIZE_PRETRAINED, IMG_SIZE_PRETRAINED)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# --- Reload datasets at the new resolution (same subset folders, no need to rebuild) ---
full_train_dataset_pt = datasets.ImageFolder(os.path.join(SUBSET_DIR, "train"), transform=train_transforms_pt)
test_dataset_pt = datasets.ImageFolder(os.path.join(SUBSET_DIR, "test"), transform=eval_transforms_pt)

from constants import RANDOM_SEED

generator = torch.Generator().manual_seed(RANDOM_SEED)
val_size = int(len(full_train_dataset_pt) * 0.1)
train_size = len(full_train_dataset_pt) - val_size
train_dataset_pt, val_dataset_pt = random_split(full_train_dataset_pt, [train_size, val_size], generator=generator)

BATCH_SIZE = 32
train_loader_pt = DataLoader(train_dataset_pt, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader_pt = DataLoader(val_dataset_pt, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader_pt = DataLoader(test_dataset_pt, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# --- Load pretrained ResNet18 ---
resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# --- Freeze all existing layers — we don't want to retrain what it already knows ---
for param in resnet.parameters():
    param.requires_grad = False

# --- Replace the final classification layer with one sized for our 15 classes ---
# resnet.fc is the original layer trained for ImageNet's 1000 classes — swap it out
num_features = resnet.fc.in_features  # 512 for ResNet18
resnet.fc = nn.Linear(num_features, 15)  # only this new layer will actually train

resnet = resnet.to(device)

# --- Loss and optimizer — only the new fc layer has requires_grad=True, so only it updates ---
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(resnet.fc.parameters(), lr=0.0001)  # note: only passing resnet.fc.parameters()

# --- Save/load setup ---
MODEL_DIR = "D:/CitrusBits/pythonic-rebirth/models"
RESNET_PATH = os.path.join(MODEL_DIR, "food101_resnet18_transfer.pth")

EPOCHS = 8

if os.path.exists(RESNET_PATH):
    print(f"Found existing model at {RESNET_PATH}, loading instead of retraining...")
    resnet.load_state_dict(torch.load(RESNET_PATH, map_location=device))
else:
    print("No saved model found, training transfer learning model...")
    for epoch in range(EPOCHS):
        resnet.train()
        running_loss = 0.0
        correct = 0
        total_samples = 0

        for images, labels in train_loader_pt:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = resnet(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

        epoch_loss = running_loss / total_samples
        epoch_acc = correct / total_samples

        resnet.eval()
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for images, labels in val_loader_pt:
                images, labels = images.to(device), labels.to(device)
                outputs = resnet(images)
                _, predicted = torch.max(outputs, 1)
                val_correct += (predicted == labels).sum().item()
                val_total += labels.size(0)
        val_acc = val_correct / val_total

        print(
            f"Epoch {epoch + 1}/{EPOCHS} — Train Loss: {epoch_loss:.4f}, Train Acc: {epoch_acc:.4f}, Val Acc: {val_acc:.4f}")

    torch.save(resnet.state_dict(), RESNET_PATH)
    print(f"Model saved to {RESNET_PATH}")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\Hp Pavilion 13 Aero/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:02<00:00, 17.6MB/s]


No saved model found, training transfer learning model...
Epoch 1/8 — Train Loss: 2.7521, Train Acc: 0.0941, Val Acc: 0.1267
Epoch 2/8 — Train Loss: 2.5545, Train Acc: 0.1763, Val Acc: 0.2267
Epoch 3/8 — Train Loss: 2.4063, Train Acc: 0.2796, Val Acc: 0.3267
Epoch 4/8 — Train Loss: 2.2605, Train Acc: 0.3841, Val Acc: 0.4033
Epoch 5/8 — Train Loss: 2.1399, Train Acc: 0.4456, Val Acc: 0.4367
Epoch 6/8 — Train Loss: 2.0338, Train Acc: 0.4915, Val Acc: 0.4733
Epoch 7/8 — Train Loss: 1.9246, Train Acc: 0.5363, Val Acc: 0.5100
Epoch 8/8 — Train Loss: 1.8238, Train Acc: 0.5778, Val Acc: 0.5533
Model saved to D:/CitrusBits/pythonic-rebirth/models\food101_resnet18_transfer.pth


In [15]:
from sklearn.metrics import classification_report, confusion_matrix

resnet.eval()
correct = 0
total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader_pt:
        images, labels = images.to(device), labels.to(device)
        outputs = resnet(images)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_acc = correct / total
print(f"Test Accuracy: {test_acc:.4f}")

class_names = full_train_dataset_pt.classes

print(classification_report(all_labels, all_preds, target_names=class_names))
print(confusion_matrix(all_labels, all_preds))

Test Accuracy: 0.5800
                     precision    recall  f1-score   support

          apple_pie       0.60      0.20      0.30        60
       caesar_salad       0.71      0.48      0.57        60
      chicken_wings       0.63      0.65      0.64        60
             donuts       0.51      0.87      0.65        60
       french_fries       0.66      0.80      0.72        60
          hamburger       0.63      0.40      0.49        60
          ice_cream       0.53      0.50      0.51        60
           pancakes       0.45      0.57      0.50        60
              pizza       0.78      0.72      0.75        60
              ramen       0.44      0.88      0.59        60
spaghetti_bolognese       0.79      0.75      0.77        60
              steak       0.54      0.55      0.55        60
              sushi       0.64      0.48      0.55        60
              tacos       0.42      0.27      0.33        60
            waffles       0.64      0.58      0.61        60

